# ArmForge control panel

**UR5e + Robotiq 2F-85 in native MuJoCo, learned with TQC-SAC + hindsight replay.**
This notebook is the project's control panel: it sets the configuration, checks the physics scene, trains, monitors,
evaluates, renders the *failing -> learning -> solved* media and regenerates the repository README. All code lives in
`src/armforge/`; the cells only configure and call it.

| step | cell | what it does |
|:--|:--|:--|
| 0 | Setup | clone/install, EGL rendering, optional Google Drive for checkpoints |
| 1 | Configure | form fields -> a validated `RunConfig` (any config key can be overridden) |
| 2 | Scene check | renders every camera, prints physics diagnostics |
| 3 | Expert baseline | scripted controller success rate (the upper-bound reference) |
| 4 | Train | foreground or background, resumable |
| 5 | Monitor | live learning curves from `metrics.csv` |
| 6 | Evaluate | held-out episodes at the full goal range |
| 7 | Media | GIFs of early failures -> learning -> final policy, stills, plots |
| 8 | Ablate (optional) | TQC vs twin-Q, HER on/off, on one plot |
| 9 | Publish | regenerate `README.md` (legacy README kept below) and commit, push, or zip (checkpoints + metrics + media) |

**Runtime:** `Runtime -> Change runtime type -> GPU` (a T4 is enough; the networks are small, the simulator is CPU-bound).

In [ ]:
#@title 0 · Setup (run once per session)
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/rezaxr14/ArmLab_Robot_RL.git" #@param {type:"string"}
REPO_REF = "main" #@param {type:"string"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}

IN_COLAB = "google.colab" in sys.modules

def sh(*cmd, check=True):
    """Run a command and stream its output into the cell (a bare subprocess.run is invisible in Colab)."""
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"command failed (exit code {code}): {' '.join(cmd)}")
    return code

if IN_COLAB:
    root = pathlib.Path("/content/ArmLab_Robot_RL")
    if not root.exists():
        _ = sh("git", "clone", "-q", "--branch", REPO_REF, REPO_URL, str(root))
    os.chdir(root)
    # EGL gives GPU-accelerated headless rendering for MuJoCo; OSMesa is the software fallback.
    _ = sh("apt-get", "-qq", "install", "-y", "libegl1", "libgl1", "libglew-dev", "libosmesa6-dev", check=False)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    os.chdir(root)
REPO_ROOT = pathlib.Path.cwd()

_ = sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]")
os.environ.setdefault("MUJOCO_GL", "egl")

# This kernel started before the editable install existed, so it cannot see the new package on its own
# (fresh processes can). Put src/ on the path and verify, instead of requiring a runtime restart.
import importlib
SRC_DIR = str(REPO_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()
import armforge
print("armforge", armforge.__version__, "imported from", armforge.__file__)

RUNS_DIR = str(REPO_ROOT / "runs")
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_DIR = "/content/drive/MyDrive/armforge_runs"   # survives runtime disconnects
os.makedirs(RUNS_DIR, exist_ok=True)
print("repo:", REPO_ROOT, "| runs:", RUNS_DIR)

_ = sh(sys.executable, "-m", "armforge", "doctor", check=False)   # prints one [ok]/[FAIL] line per check

In [ ]:
#@title 1 · Configure the experiment
#@markdown **Preset** (a YAML in `configs/`). Everything below overrides it; leave a field at its "keep" value to use the preset's.
PRESET = "pickplace" #@param ["reach", "push", "pickplace", "pickplace_clutter", "smoke"]
RUN_NAME = "" #@param {type:"string"}
SEED = 0 #@param {type:"integer"}
DEVICE = "preset" #@param ["preset", "auto", "cuda", "cpu"]
TOTAL_STEPS = -1 #@param {type:"integer"}
NUM_ENVS = -1 #@param {type:"integer"}  # -1 = preset, 0 = one per CPU core
#@markdown ---
#@markdown **Algorithm**
CRITIC = "preset" #@param ["preset", "tqc", "twin"]
CRITIC_ENSEMBLE = -1 #@param {type:"integer"}
N_QUANTILES = -1 #@param {type:"integer"}
TOP_QUANTILES_DROPPED = -1 #@param {type:"integer"}
HER_PROB = -1.0 #@param {type:"number"}
UPDATES_PER_STEP = -1.0 #@param {type:"number"}
BATCH_SIZE = -1 #@param {type:"integer"}
#@markdown **Physics** (friction values are MuJoCo sliding friction)
PAD_FRICTION = -1.0 #@param {type:"number"}
TABLE_FRICTION = -1.0 #@param {type:"number"}
CUBE_FRICTION = -1.0 #@param {type:"number"}
CUBE_MASS_GRAMS = -1.0 #@param {type:"number"}
ARM_JOINT_DAMPING = -1.0 #@param {type:"number"}
DOMAIN_RANDOMIZATION = "preset" #@param ["preset", "on", "off"]
#@markdown **Scene**
TRAY = "preset" #@param ["preset", "on", "off"]
DISTRACTORS = -1 #@param {type:"integer"}
#@markdown **Learning aids**
CURRICULUM = "preset" #@param ["preset", "on", "off"]
DEMO_EPISODES = -1 #@param {type:"integer"}
#@markdown Any other config key, as a Python dict with dotted keys, e.g. `{"env.task.horizon": 100}`
EXTRA_OVERRIDES = {}

import json, yaml
from armforge.config import RunConfig, apply_overrides, from_dict, load_config, save_config, to_dict

overrides = {}
def set_if(key, value, keep):
    if value != keep:
        overrides[key] = value
def tri(value):
    return {"on": True, "off": False}.get(value, "preset")

set_if("train.seed", SEED, None)
set_if("train.device", DEVICE, "preset")
set_if("train.total_steps", TOTAL_STEPS, -1)
set_if("train.num_envs", NUM_ENVS, -1)
set_if("algo.critic", CRITIC, "preset")
set_if("algo.critic_ensemble", CRITIC_ENSEMBLE, -1)
set_if("algo.n_quantiles", N_QUANTILES, -1)
set_if("algo.top_quantiles_to_drop", TOP_QUANTILES_DROPPED, -1)
set_if("algo.her_prob", HER_PROB, -1.0)
set_if("algo.updates_per_step", UPDATES_PER_STEP, -1.0)
set_if("algo.batch_size", BATCH_SIZE, -1)
set_if("env.physics.pad_friction", PAD_FRICTION, -1.0)
set_if("env.physics.table_friction", None if TABLE_FRICTION < 0 else [TABLE_FRICTION, 0.005, 0.0001], None)
set_if("env.physics.object_friction", None if CUBE_FRICTION < 0 else [CUBE_FRICTION, 0.01, 0.0005], None)
set_if("env.physics.object_mass", None if CUBE_MASS_GRAMS < 0 else CUBE_MASS_GRAMS / 1000.0, None)
set_if("env.physics.arm_joint_damping", ARM_JOINT_DAMPING, -1.0)
set_if("env.physics.randomize.enabled", tri(DOMAIN_RANDOMIZATION), "preset")
set_if("env.scene.tray", tri(TRAY), "preset")
set_if("env.scene.distractors", DISTRACTORS, -1)
set_if("train.curriculum.enabled", tri(CURRICULUM), "preset")
set_if("train.demos.episodes", DEMO_EPISODES, -1)
overrides = {k: v for k, v in overrides.items() if v is not None}
overrides.update(EXTRA_OVERRIDES)

cfg = load_config(f"configs/{PRESET}.yaml", overrides)
cfg.name = RUN_NAME or f"{PRESET}-s{cfg.train.seed}"
cfg.out_dir = RUNS_DIR
if cfg.train.num_envs <= 0:
    cfg.train.num_envs = max(1, os.cpu_count() or 1)

RUN_DIR = pathlib.Path(cfg.out_dir) / cfg.name
buf_gb = cfg.train.buffer_episodes * (cfg.env.task.horizon + 1) * 40 * 4 / 1e9
print(f"run: {cfg.name}   task: {cfg.env.task.name}   steps: {cfg.train.total_steps:,}   workers: {cfg.train.num_envs}")
print(f"overrides applied: {json.dumps(overrides, default=str)}")
print(f"replay buffer (GPU/CPU RAM): ~{buf_gb:.2f} GB")
print(yaml.safe_dump(to_dict(cfg), sort_keys=False))

In [ ]:
#@title 2 · Scene check: cameras and physics diagnostics
from dataclasses import replace
import numpy as np, mujoco
from PIL import Image
from IPython.display import display
from armforge.env.arm_env import ArmForgeEnv
from armforge.scene import CAMERAS

env = ArmForgeEnv(replace(cfg.env, seed=1_000_000))
env.reset(seed=1_000_000)
m, d = env.model, env.data
total_mass = float(m.body_mass.sum())
print(f"bodies {m.nbody} | geoms {m.ngeom} | joints {m.njnt} | actuators {m.nu} | equality constraints {m.neq}")
print(f"timestep {m.opt.timestep*1000:.1f} ms x {env.substeps} substeps = {1/env.dt:.0f} Hz control | cone: {mujoco.mjtCone(m.opt.cone).name}")
print(f"total mass {total_mass:.2f} kg | contacts at reset: {d.ncon} | obs dim {env.obs_dim} | action dim {env.action_dim}")
tbl = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, "table_top")
cube = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, "cube_geom")
print("table friction", m.geom_friction[tbl], "| cube friction", m.geom_friction[cube], "| cube mass", m.body_mass[env.cube_body])
print("arm actuator kp:", m.actuator_gainprm[env.arm_act, 0], "| force limits:", m.actuator_forcerange[env.arm_act, 1])

tiles = [Image.fromarray(env.render(cam, 420, 315)) for cam in ("front", "side", "top", "close")]
sheet = Image.new("RGB", (840, 630))
for i, t in enumerate(tiles):
    sheet.paste(t, ((i % 2) * 420, (i // 2) * 315))
display(sheet)
env.close()

In [ ]:
#@title 3 · Scripted-expert baseline (the reference the learner is judged against)
RUN_EXPERT = True #@param {type:"boolean"}
EXPERT_EPISODES = 20 #@param {type:"integer"}
if RUN_EXPERT:
    from collections import Counter
    from armforge.rl.expert import rollout_expert, summarize_episode
    env = ArmForgeEnv(replace(cfg.env, seed=1_000_000))
    eps = [rollout_expert(env, seed=1_000_000 + i) for i in range(EXPERT_EPISODES)]
    wins = sum(e["success"] for e in eps)
    print(f"scripted expert on '{cfg.env.task.name}': {wins}/{EXPERT_EPISODES} episodes succeeded")
    for e in eps:
        print(summarize_episode(e))
    stuck = Counter(e["trace"][-1]["phase"] for e in eps if not e["success"])
    if stuck:
        print("failed episodes ended in phase:", dict(stuck))
    if wins < EXPERT_EPISODES // 2:
        print("WARNING: the expert is failing - check the scene/friction settings before training with demonstrations.")
        print("Paste the lines above (one per episode) when reporting this; they show the phase each episode got stuck in.")
    bad = [e for e in eps if not e["success"]]
    if bad:
        import json
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        (RUN_DIR / "expert_failures.json").write_text(json.dumps([{k: e[k] for k in ("seed", "phases", "trace", "goal_height")} for e in bad]))
        for e in bad[:2]:  # step table of the first failures: time, phase, hand, cube, gripper, pad contacts
            print(f"--- seed {e['seed']} (every 2nd step) ---  t phase | hand xyz | cube xyz | grip pads")
            for r in e["trace"][::2]:
                print(f"{r['t']:3d} {r['phase']:8s} | {r['ee']} | {r['ag']} | {r['grip']:.2f} {r['pads']}")
    env.close()

In [ ]:
#@title 3b · Forward-kinematics check: CPU numpy vs MuJoCo (independent implementations of the same FK)
RUN_FK_CHECK = True #@param {type:"boolean"}
FK_RANDOM_CONFIGS = 1000 #@param {type:"integer"}
FK_EXPERT_EPISODES = 3 #@param {type:"integer"}
if RUN_FK_CHECK:
    from armforge.fkcheck import benchmark, compare_random, compare_trajectory
    rand = compare_random(cfg.env, samples=FK_RANDOM_CONFIGS)
    print(rand.table())
    ok = rand.passed()
    if FK_EXPERT_EPISODES:
        traj, gap = compare_trajectory(cfg.env, episodes=FK_EXPERT_EPISODES)
        print(traj.table())
        print(f"environment ee_pos vs CPU FK of the same joint angles: max {gap:.6f} mm")
        ok = ok and traj.passed() and gap * 1e-3 <= 1e-6
    t = benchmark(cfg.env)
    print(f"time per FK call: CPU numpy {t['cpu_numpy']:.0f} us | MuJoCo {t['mujoco']:.1f} us")
    print("FK CHECK", "PASSED: the numpy walker and MuJoCo agree" if ok else "FAILED: the model's kinematics differ between the two - investigate before trusting any result")

In [ ]:
#@title 4 · Train
BACKGROUND = False #@param {type:"boolean"}
RESUME = True #@param {type:"boolean"}
#@markdown Background mode runs training as a separate process so you can run the Monitor cell while it trains.
launch_cfg = RUN_DIR / "launch.yaml"
save_config(cfg, launch_cfg)

if BACKGROUND:
    log = open(RUN_DIR / "train.log", "a")
    cmd = [sys.executable, "-m", "armforge", "train", "--config", str(launch_cfg)] + (["--resume"] if RESUME else [])
    TRAIN_PROC = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    print("started pid", TRAIN_PROC.pid, "| log:", RUN_DIR / "train.log")
else:
    from armforge.rl.trainer import Trainer
    trainer = Trainer(cfg, resume=RESUME)
    try:
        summary = trainer.train()
    finally:
        trainer.close()
    print(json.dumps(summary, indent=2))

In [ ]:
#@title 5 · Monitor (live learning curves)
MONITOR_SECONDS = 0 #@param {type:"integer"}
#@markdown 0 = plot once. Otherwise refresh every 30 s for that long (use with background training).
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output
from armforge.report import read_metrics

def show():
    rows = read_metrics(RUN_DIR / "metrics.csv")
    if not rows:
        print("no metrics yet"); return
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    ev = [r for r in rows if r.get("eval_success") is not None]
    tr = [r for r in rows if r.get("train_success") is not None]
    ax[0].plot([r["steps"] for r in ev], [r["eval_success"] for r in ev], "o-", color="#2a78d6", label="eval (full range)")
    ax[0].plot([r["steps"] for r in tr], [r["train_success"] for r in tr], color="#eb6834", label="train (curriculum)")
    ax[0].set_ylim(-0.02, 1.02); ax[0].set_title("success rate"); ax[0].legend(frameon=False)
    ax[1].plot([r["steps"] for r in ev], [100 * r["eval_final_distance"] for r in ev], "o-", color="#2a78d6"); ax[1].set_title("final goal distance (cm)")
    cl = [r for r in rows if r.get("critic_loss") is not None]
    ax[2].plot([r["steps"] for r in cl], [r["critic_loss"] for r in cl], color="#2a78d6"); ax[2].set_title("critic loss")
    for a in ax: a.set_xlabel("env steps"); a.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    last = rows[-1]
    print(f"step {int(last['steps']):,} | sps {last['sps']} | goal range {last['goal_scale']} | {last['time_s']/60:.1f} min")

end = time.time() + MONITOR_SECONDS
while True:
    clear_output(wait=True); show()
    if time.time() >= end: break
    time.sleep(30)

In [ ]:
#@title 6 · Evaluate on held-out seeds (full goal range, deterministic policy)
EVAL_CHECKPOINT = "best.pt" #@param ["best.pt", "final.pt"]
EVAL_EPISODES = 100 #@param {type:"integer"}
import pandas as pd
from armforge.media import load_policy
from armforge.rl.evaluate import agent_policy, evaluate_policy

agent = load_policy(RUN_DIR / EVAL_CHECKPOINT, cfg, "cpu")
res = evaluate_policy(cfg.env, agent_policy(agent), EVAL_EPISODES)
rows = pd.DataFrame(res.pop("rows"))
print(json.dumps(res, indent=2))
display(rows.describe().T[["mean", "min", "max"]])

In [ ]:
#@title 7 · Render the failing -> learning -> solved media
CAMERAS_TO_RENDER = "hero" #@param ["hero", "front", "side", "close"]
CLIPS_PER_STAGE = 2 #@param {type:"integer"}
STATS_EPISODES = 30 #@param {type:"integer"}
from IPython.display import Image as IPImage
from armforge.report import publish_task, render_scene_gallery

manifest = publish_task(RUN_DIR, REPO_ROOT, cameras=(CAMERAS_TO_RENDER,), clips_per_stage=CLIPS_PER_STAGE, stats_episodes=STATS_EPISODES, out_name=PRESET)
render_scene_gallery(cfg, REPO_ROOT / "docs" / "media" / "scene_gallery.png")
media = REPO_ROOT / "docs" / "media" / cfg.env.task.name
print(pd.DataFrame(manifest["stages"])[["step", "label", "success_rate", "mean_final_distance"]].to_string(index=False))
display(IPImage(filename=str(media / "progression.gif")))
display(IPImage(filename=str(media / "stages_end.png")))
display(IPImage(filename=str(media / "learning_curve.png")))

In [ ]:
#@title 8 · (Optional) ablation: TQC vs twin-Q, and HER on/off
RUN_ABLATION = False #@param {type:"boolean"}
ABLATION_STEPS = 150000 #@param {type:"integer"}
if RUN_ABLATION:
    from armforge.report import plot_comparison
    from armforge.rl.trainer import Trainer
    variants = {
        "TQC + HER": {},
        "twin-Q SAC + HER": {"algo.critic": "twin"},
        "TQC, no HER": {"algo.her_prob": 0.0},
    }
    base = to_dict(cfg)
    dirs = []
    for label, extra in variants.items():
        name = f"{cfg.name}-abl-" + label.lower().replace(" ", "").replace(",", "").replace("+", "")
        c = from_dict(RunConfig, apply_overrides(base, {**extra, "name": name, "train.total_steps": ABLATION_STEPS}))
        t = Trainer(c)
        try:
            t.train()
        finally:
            t.close()
        dirs.append(pathlib.Path(c.out_dir) / name)
    out = plot_comparison(dirs, list(variants), REPO_ROOT / "docs" / "media" / cfg.env.task.name / "ablation.png", f"{cfg.env.task.name}: ablation ({ABLATION_STEPS:,} steps, one seed)")
    display(IPImage(filename=str(out)))

In [ ]:
#@title 9 · Publish: regenerate README.md and save the work
ACTION = "preview README only" #@param ["preview README only", "git commit", "git commit and push", "zip for download"]
INCLUDE_REPLAY_BUFFER = False #@param {type:"boolean"}
GIT_USER_NAME = "" #@param {type:"string"}
GIT_USER_EMAIL = "" #@param {type:"string"}
from armforge.readme import build_readme

legacy = (REPO_ROOT / "legacy" / "README.md").read_text(encoding="utf-8")
readme = build_readme(legacy, REPO_ROOT / "docs" / "media")
(REPO_ROOT / "README.md").write_text(readme, encoding="utf-8")
print(f"README.md regenerated ({len(readme.splitlines())} lines; legacy README kept in full below the new section)")

if ACTION.startswith("git"):
    import os, shutil
    if GIT_USER_NAME:  sh("git", "config", "user.name", GIT_USER_NAME)
    if GIT_USER_EMAIL: sh("git", "config", "user.email", GIT_USER_EMAIL)
    sh("git", "add", "README.md", "docs/media")
    # Publish the learned weights too: best/final plus every milestone snapshot, copied out of
    # runs/ (gitignored) into a small tracked directory.
    weights = REPO_ROOT / "weights" / cfg.name
    weights.mkdir(parents=True, exist_ok=True)
    for w in [RUN_DIR / "best.pt", RUN_DIR / "final.pt", *sorted((RUN_DIR / "milestones").glob("*.pt"))]:
        if w.exists():
            shutil.copy2(w, weights / w.name)
    sh("git", "add", "-f", "weights")
    sh("git", "commit", "-m", f"docs: publish {cfg.env.task.name} results + weights (run {cfg.name})", check=False)
    if ACTION == "git commit and push":
        # Locally the remote is already authenticated (SSH key / gh). In Colab, set GH_TOKEN or
        # let the prompt ask for a token; the token is never written to disk.
        token = os.environ.get("GH_TOKEN") or os.environ.get("GITHUB_TOKEN")
        if token:
            url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
            subprocess.run(["git", "push", url, f"HEAD:{REPO_REF}"], check=True)
        elif IN_COLAB:
            import getpass
            token = getpass.getpass("GitHub token with repo write access (not stored): ")
            url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
            subprocess.run(["git", "push", url, f"HEAD:{REPO_REF}"], check=True)
        else:
            sh("git", "push", "origin", f"HEAD:{REPO_REF}")
elif ACTION == "zip for download":
    from armforge.archive import pack_run
    info = pack_run(RUN_DIR, "/content/armforge_run.zip" if IN_COLAB else "armforge_run.zip", repo_root=REPO_ROOT, include_buffer=INCLUDE_REPLAY_BUFFER)
    print(f"wrote {info['zip']}: {info['files']} files, {info['bytes'] / 1e6:.1f} MB (checkpoints + metrics + media)")
    if IN_COLAB:
        from google.colab import files
        files.download(info["zip"])